# Détection de viennoiseries avec YOLOv8
## Groupe 5 - Parcours IMM - ENSEEIHT 2025-2026
Ahmed EL MAZROUA, Amina KAMAL, Mohammed RADHI

Ce notebook présente l'entraînement et l'évaluation d'un modèle YOLOv8n pour la détection automatique de 7 classes de viennoiseries/pâtisseries françaises : chausson aux pommes, chocolatine, croissant, éclair, macaron, millefeuille, tarte.


## 1. Installation et imports
Installation de la bibliothèque Ultralytics (YOLOv8).

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Chargement du dataset
Le dataset est hébergé sur GitHub. Il contient 780 images annotées au format
YOLO (bounding boxes), réparties en 543 train / 113 val / 124 test selon un
split stratifié 70/15/15 garantissant la représentativité de chaque classe
dans chaque sous-ensemble. Les images manuelles (150, issues de boulangeries
toulousaines) ont été prioritairement allouées au test set pour garantir une
évaluation sur des conditions réelles.

In [ ]:
!git clone https://github.com/ElMazrouaCS/viennoiseries-detection.git
%cd viennoiseries-detection

for split in ["train", "val", "test"]:
    n = len(os.listdir(f"dataset/{split}/images"))
    print(f"{split}: {n} images")
# Attendu : 543 / 113 / 124

Cloning into 'viennoiseries-detection'...
remote: Enumerating objects: 1537, done.
remote: Total 1537 (delta 0), reused 0 (delta 0), pack-reused 1537 (from 3)
Receiving objects: 100% (1537/1537), 179.27 MiB | 34.95 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/viennoiseries-detection/viennoiseries-detection
train: 543 images
val: 113 images
test: 124 images


## 3. Entraînement du modèle

**Architecture :** YOLOv8n (nano), 3 007 013 paramètres, 8.1 GFLOPs.
Architecture à une seule passe (single-stage) avec tête de détection multi-échelle
opérant sur 3 résolutions (P3/P4/P5). Poids initialisés par transfer learning
depuis COCO (319/355 couches transférées).

**Hyperparamètres :**
- Optimizer : AdamW (lr=0.000909, momentum=0.9), déterminé automatiquement
- Epochs : 50, batch=16, imgsz=640, patience=10 (early stopping)
- Augmentation : RandAugment + Albumentations (Blur, MedianBlur, CLAHE, ToGray)
  + augmentations géométriques YOLOv8 (flip, mosaic, HSV)

**Justification des choix :**
YOLOv8n a été retenu pour son équilibre vitesse/performance adapté à un dataset
de taille modeste (780 images). Le transfer learning depuis COCO accélère la
convergence en réutilisant des features visuelles génériques (textures, contours).
L'augmentation aggressive compense la taille limitée du dataset.

In [ ]:
model = YOLO("yolov8n.pt")
model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    augment=True,
    name="viennoiseries_v1"
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=viennoiseries_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspectiv

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 626.4±501.7 MB/s, size: 134.4 KB)
val: Scanning /content/viennoiseries-detection/viennoiseries-detection/dataset/val/labels... 113 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 113/113 1.6Kit/s 0.1s
val: New cache created: /content/viennoiseries-detection/viennoiseries-detection/dataset/val/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000909, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /content/viennoiseries-detection/viennoiseries-detection/runs/detect/viennoiseries_v1/labels.jpg... 
Image sizes 640 train, 6

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78ba55986b70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

## 4. Évaluation sur le test set
Évaluation du meilleur modèle (best.pt, sélectionné sur mAP50 validation)
sur les 124 images du test set, jamais vues pendant l'entraînement.

In [ ]:
model = YOLO("runs/detect/viennoiseries_v1/weights/best.pt")
test_results = model.val(data="data.yaml", split="test")

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2256.7±691.8 MB/s, size: 216.5 KB)
val: Scanning /content/viennoiseries-detection/viennoiseries-detection/dataset/test/labels... 124 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 124/124 1.9Kit/s 0.1s
val: New cache created: /content/viennoiseries-detection/viennoiseries-detection/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.6it/s 4.9s
                   all        124        378      0.706      0.718      0.745      0.452
              chausson         19         57       0.87      0.842      0.901      0.705
           chocolatine         14         80      0.752        0.8      0.854      0.632
             croissant         18         83      0.548      0.807      0.757    

## 5. Prédictions visuelles
Génération des prédictions sur l'ensemble du test set (seuil de confiance : 0.25).
Les images annotées sont sauvegardées dans runs/detect/test_predictions/.

In [ ]:
model.predict(
    source="dataset/test/images",
    save=True,
    conf=0.25,
    name="test_predictions"
)


image 1/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000004.jpg: 640x640 1 chausson, 7.8ms
image 2/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000005.jpg: 448x640 1 chausson, 38.5ms
image 3/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000012.jpg: 512x640 1 chausson, 38.7ms
image 4/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000014.jpg: 640x640 1 chausson, 10.0ms
image 5/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000015.jpg: 480x640 4 chaussons, 37.9ms
image 6/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000018.jpg: 640x640 1 chausson, 7.9ms
image 7/124 /content/viennoiseries-detection/viennoiseries-detection/dataset/test/images/chausson_000029.jpg: 448x640 3 chaussons, 6.8ms
image 8/124 /content/viennoiseries-detect

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'chausson', 1: 'chocolatine', 2: 'croissant', 3: 'eclair', 4: 'macaron', 5: 'millefeuille', 6: 'tarte'}
 obb: None
 orig_img: array([[[216, 214, 214],
         [216, 214, 214],
         [216, 214, 214],
         ...,
         [210, 208, 208],
         [210, 208, 208],
         [210, 208, 208]],
 
        [[216, 214, 214],
         [216, 214, 214],
         [216, 214, 214],
         ...,
         [210, 208, 208],
         [210, 208, 208],
         [210, 208, 208]],
 
        [[216, 214, 214],
         [216, 214, 214],
         [216, 214, 214],
         ...,
         [210, 208, 208],
         [210, 208, 208],
         [210, 208, 208]],
 
        ...,
 
        [[223, 221, 221],
         [222, 220, 220],
         [222, 220, 220],
         ...,
         [222, 220, 220],
         [222, 220, 220],
         [222, 220, 220]],
 
        [[222, 2

## 6. Analyse des résultats

### Résultats quantitatifs (test set)
| Classe       | Précision | Rappel | mAP50 | mAP50-95 |
|---|---|---|---|---|
| chausson     | 0.870 | 0.842 | 0.901 | 0.705 |
| chocolatine  | 0.752 | 0.800 | 0.854 | 0.632 |
| croissant    | 0.548 | 0.807 | 0.757 | 0.500 |
| éclair       | 0.739 | 0.579 | 0.579 | 0.277 |
| macaron      | 0.350 | 0.176 | 0.271 | 0.109 |
| millefeuille | 0.773 | 0.944 | 0.949 | 0.527 |
| tarte        | 0.910 | 0.877 | 0.902 | 0.416 |
| **all**      | **0.706** | **0.718** | **0.745** | **0.452** |

**Cas macaron :** La classe macaron présente un recall anormalement bas (0.176 test)
attribuable à des annotations incomplètes : les images contiennent fréquemment
10-20+ macarons dont seule une fraction (1-2) avait été annotée, constituant
un signal d'entraînement contradictoire. Cette classe est écartée de l'analyse
comparative sur décision de l'encadrant. Sur les 6 classes restantes :
mAP50 ≈ 0.824.

### Analyse qualitative
**Points forts :** chausson (multi-instances haute confiance 0.92-0.97),
millefeuille et tarte (formes distinctives bien apprises), chocolatine
(robuste même en scène dense de boulangerie).

**Limites observées :** confusion chocolatine/croissant sur images mixtes
(morphologie similaire, seule la chocolatine visible distingue les deux),
faux positif chausson sur éclair traditionnel non glacé (forme allongée
et couleur dorée similaires), sous-détection macaron confirmant le problème
d'annotation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/viennoiseries_projet
!cp -r runs/detect/viennoiseries_v1 /content/drive/MyDrive/viennoiseries_projet/
!cp -r runs/detect/test_predictions /content/drive/MyDrive/viennoiseries_projet/

Mounted at /content/drive
